## **1. Objetivo General del Rol (Negocio)**

Actuar como consultora especializada en analítica para **StreamView Analytics** con el fin de **comprender el comportamiento de los usuarios y apoyar la toma de decisiones estratégicas** en cuatro frentes clave:

- **Retención** de clientes.

- Nivel de interacción (**engagement**).

- Preferencias de **consumo de contenido.**

- **Experiencia** de usuario.

## **2. Objetivos Específicos del Proyecto (Lo que deben lograr)**

- **Integrar y preparar datos:** Unificar las distintas fuentes corporativas (usuarios, suscripciones, reproducciones, dispositivos, calificaciones e interacciones) identificando las variables críticas para el análisis.

- **Explorar y detectar patrones:** Realizar un análisis exploratorio visual para encontrar tendencias, relaciones clave y hallazgos orientados al negocio.

- **Diseñar y justificar visualizaciones:** Crear gráficos pertinentes justificando la elección de encodificación visual (canales, marcas, percepción) según la audiencia.

- **Construir una herramienta analítica (Dashboard):** Implementar un panel interactivo (ej. Power BI, Tableau, Dash, Streamlit) que incluya KPIs principales, navegación fluida y filtros para la toma de decisiones.

- **Comunicar con Data Storytelling:** Estructurar una narrativa visual comprensible que guíe al espectador desde el dato hasta la conclusión, adaptando el lenguaje al perfil de la audiencia.

- **Evaluar críticamente el trabajo:** Identificar objetivamente las fortalezas, limitaciones y oportunidades de mejora de la solución implementada.

- **Formular recomendaciones accionables:** Proponer decisiones concretas basadas estrictamente en la evidencia de los datos analizados.

- **Garantizar la reproducibilidad técnica:** Entregar el código, datos y estructura de carpetas (data/, notebooks/, dashboard/, src/, README.md) listos para que cualquier persona pueda replicar el proyecto sin errores.

### Limpieza basica y cruzamiento de datasets

In [3]:
import pandas as pd

# Cargamos cada fuente en un DataFrame separado para poder revisar y limpiar
# las peliculas y las series con el mismo procedimiento.
df_movies_raw = pd.read_csv('../data/netflix_movies_detailed_up_to_2025.csv')
df_tv_shows_raw = pd.read_csv('../data/netflix_tv_shows_detailed_up_to_2025.csv')


def limpiar_catalogo(df, fuente):
    """Aplica una limpieza basica y agrega campos comunes al catalogo."""
    df = df.copy()

    # Estandarizamos los nombres de columnas para evitar diferencias de escritura.
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
    )

    # Convertimos a texto antes de limpiar para tolerar columnas completamente vacias.
    columnas_texto = df.select_dtypes(include=['object', 'string']).columns
    for columna in columnas_texto:
        df[columna] = df[columna].astype('string').str.strip()
        df[columna] = df[columna].replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

    # Convertimos las columnas de fecha y numericas para facilitar el analisis.
    if 'date_added' in df.columns:
        df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

    columnas_numericas = [
        'show_id', 'release_year', 'rating', 'popularity',
        'vote_count', 'vote_average', 'budget', 'revenue'
    ]
    for columna in columnas_numericas:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors='coerce')

    # Conservamos el origen y creamos una llave unica para el catalogo combinado.
    df['source_dataset'] = fuente
    df['content_type'] = df['type'].astype('string').str.strip().str.lower()
    df['content_key'] = fuente + '_' + df['show_id'].astype('Int64').astype(str)

    # Separamos duration en un valor numerico y una unidad (minutos o temporadas).
    if 'duration' in df.columns:
        duration_texto = df['duration'].astype('string')
        partes_duration = duration_texto.str.extract(
            r'(?P<duration_value>\d+)\s*(?P<duration_unit>.*)'
        )
        df['duration_value'] = pd.to_numeric(partes_duration['duration_value'], errors='coerce')
        df['duration_unit'] = partes_duration['duration_unit'].str.strip().replace('', pd.NA)

    # Un show_id repetido dentro de una misma fuente representa un registro duplicado.
    df = df.drop_duplicates(subset='show_id', keep='first').reset_index(drop=True)
    return df


# Limpiamos ambas fuentes con la misma funcion para que sean comparables.
df_movies = limpiar_catalogo(df_movies_raw, 'movies')
df_tv_shows = limpiar_catalogo(df_tv_shows_raw, 'tv_shows')

# Como las columnas adicionales no son iguales en ambas fuentes, concat permite
# conservar toda la informacion y completa con NaN donde una fuente no tiene un campo.
df_catalogo = pd.concat(
    [df_movies, df_tv_shows],
    ignore_index=True,
    sort=False
)

# Validaciones simples para comprobar que el cruce genero un marco coherente.
assert len(df_catalogo) == len(df_movies) + len(df_tv_shows)
assert df_catalogo['content_key'].is_unique

print(f'Peliculas limpias: {df_movies.shape[0]:,} filas')
print(f'Series limpias: {df_tv_shows.shape[0]:,} filas')
print(f'Catalogo combinado: {df_catalogo.shape[0]:,} filas y {df_catalogo.shape[1]:,} columnas')
print('\nRegistros por tipo:')
print(df_catalogo['content_type'].value_counts(dropna=False))

# Este reporte permite revisar rapidamente los faltantes antes del analisis.
reporte_faltantes = (
    df_catalogo.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename('porcentaje_faltante')
    .to_frame()
)
reporte_faltantes.head(10)

Peliculas limpias: 16,000 filas
Series limpias: 15,991 filas
Catalogo combinado: 31,991 filas y 23 columnas

Registros por tipo:
content_type
movie      16000
tv show    15991
Name: count, dtype: Int64


,porcentaje_faltante
duration_unit,50.014066
duration_value,50.014066
duration,50.014066
budget,49.985934
revenue,49.985934
director,34.672252
description,10.431059
country,7.067613
cast,4.251196
genres,3.372824
